# 03 · 手写数字：从 sklearn 到 PyTorch

前两个 notebook 用 sklearn 的 `.fit()` 一行搞定。但神经网络的实际用法是另一套逻辑：**自己写训练循环**。

这个 notebook 用同一份数据（8×8 手写数字）走两遍：

- 第一遍 sklearn 的 MLP——看清效果上限
- 第二遍 PyTorch——理解 `loss.backward()` 这一步到底在做什么

你的环境装的是 **CPU 版 torch**（`2.15.0+cpu`）。这份数据极小，CPU 跑几秒就完，不需要显卡。

In [ ]:
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "WenQuanYi Zen Hei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("中文字体:", matplotlib.rcParams["font.sans-serif"][0])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target
print("数据形状:", X.shape, "(1797 张 8x8 图，展平成 64 维)")
print("像素范围:", X.min(), "~", X.max())

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(label), fontsize=9)
    ax.axis("off")
plt.suptitle("前 16 张样本（灰度反转，深色是笔画）")
plt.tight_layout()
plt.show()

## 第一遍：sklearn 的 MLP

`MLPClassifier` 就是多层感知机（全连接神经网络）。注意两点：

1. **像素要归一化到 0–1**，否则梯度会爆
2. `hidden_layer_sizes=(128,)` 表示一个隐层、128 个神经元

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X / 16.0, y, test_size=0.2, random_state=42, stratify=y
)

mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
print(f"测试集准确率: {mlp.score(X_test, y_test):.3f}")

## 第二遍：PyTorch 手写训练循环

这是**每个 PyTorch 项目都一模一样的骨架**，五步，背下来：

| 步骤 | 代码 | 作用 |
|---|---|---|
| 1 前向 | `pred = model(x)` | 算出预测 |
| 2 算损失 | `loss = criterion(pred, y)` | 预测和答案差多少 |
| 3 清梯度 | `optimizer.zero_grad()` | **不清会累加** |
| 4 反向 | `loss.backward()` | 算出每个参数的梯度 |
| 5 更新 | `optimizer.step()` | 沿梯度走一小步 |

第 3 步最容易忘。PyTorch 默认把梯度**累加**，忘了清就等于拿上一步的梯度又加一遍，模型会莫名其妙不收敛。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用设备:", device)

Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.long)

model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)
print(model)

最后一层输出 10 个 **logits**（未归一化的分数），不接 softmax。

因为 `CrossEntropyLoss` 内部已经包含 softmax 了，模型里再加一次就是重复计算——**这是新手最常见的错误**，表现为损失一直降不下去。

In [ ]:
import time

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Xtr, ytr, Xte, yte = Xtr.to(device), ytr.to(device), Xte.to(device), yte.to(device)
history = []
t0 = time.time()

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(Xtr), ytr)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    history.append((loss.item(), acc))

    if epoch % 20 == 0 or epoch == 99:
        print(f"epoch {epoch:3d}  loss={loss.item():.4f}  测试准确率={acc:.3f}")

print(f"\n100 轮耗时 {time.time() - t0:.1f}s")

In [ ]:
losses, accs = zip(*history)
fig, ax1 = plt.subplots(figsize=(8, 3.5))
ax1.plot(losses, color="crimson", label="训练损失")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss", color="crimson")
ax2 = ax1.twinx()
ax2.plot(accs, color="steelblue", label="测试准确率")
ax2.set_ylabel("accuracy", color="steelblue")
plt.title("训练曲线")
plt.tight_layout()
plt.show()

## 看错在哪

把预测错的样本画出来。**这一步比看准确率有用得多**——错误集中的地方往往暴露的是数据问题，而不是模型问题。

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(Xte).argmax(1).cpu().numpy()

wrong = np.where(preds != y_test)[0]
print(f"错了 {len(wrong)} / {len(y_test)} 张")

frames = X_test[wrong[:8]].reshape(-1, 8, 8)
fig, axes = plt.subplots(1, max(len(frames), 1), figsize=(1.7 * max(len(frames), 1), 2.2))
for ax, img, i in zip(np.atleast_1d(axes), frames, wrong[:8]):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(f"真 {y_test[i]} / 猜 {preds[i]}", fontsize=9, color="crimson")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 留给你动手的

1. 把学习率从 `1e-3` 改成 `0.1` 和 `1e-5`，各跑一次看曲线——**学习率是第一个要调的超参数**。
2. 故意删掉 `optimizer.zero_grad()`，跑 20 轮。损失不降或者乱跳，就是梯度累加造成的。
3. 加一层 `nn.Dropout(0.3)`，看准确率是升还是降。数据小时 Dropout 常常反而有害。
4. 把隐层从 128 缩到 8，再跑——你能看到**欠拟合**的曲线长什么样：训练损失都降不下去。
5. 换成 `MNIST`（28×28，7 万张）试试。数据下载要联网，训练时间会到分钟级——那时你就能体会为什么真训练需要显卡。